In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import pickle
import matplotlib.pyplot as plt

In [3]:
# load features from previous notebook
train_features = pd.read_csv('../output/features_train.csv')
test_features = pd.read_csv('../output/features_test.csv')

print(f"Train: {train_features.shape}")
print(f"Test: {test_features.shape}")

Train: (289263, 16)
Test: (72316, 16)


In [4]:
# helpers for measuring bin quality
def calc_entropy(labels):
    if len(labels) == 0:
        return 0.0
    unique, counts = np.unique(labels, return_counts=True)
    probs = counts / len(labels)
    return -np.sum(probs * np.log2(probs + 1e-10))

def calc_purity(bins, labels):
    # how well do bins separate classes
    unique_bins = np.unique(bins)
    purities = []
    
    for b in unique_bins:
        bin_labels = labels[bins == b]
        if len(bin_labels) == 0:
            continue
        ent = calc_entropy(bin_labels)
        purity = 1 - (ent / np.log2(6))  # 6 classes
        purities.append(purity)
    
    return np.mean(purities) if purities else 0.0

In [6]:
def discretize(X_train, y_train, X_test, n_bins=5):
    # handle constant features
    if X_train.std() == 0:
        return np.zeros(len(X_train)), np.zeros(len(X_test)), None, 'constant', 0.0
    
    try:
        # use decision tree to find good splits
        tree = DecisionTreeClassifier(
            max_leaf_nodes=n_bins * 2,
            min_samples_leaf=500,
            random_state=42
        )
        tree.fit(X_train.reshape(-1, 1), y_train)
        
        # get thresholds from tree
        thresholds = tree.tree_.threshold[tree.tree_.threshold != -2]
        thresholds = np.sort(np.unique(thresholds))
        
        if len(thresholds) == 0:
            raise ValueError("no splits")
        
        # if too many thresholds, keep best ones by info gain
        if len(thresholds) > n_bins - 1:
            gains = []
            for t in thresholds:
                left = y_train[X_train <= t]
                right = y_train[X_train > t]
                
                total = calc_entropy(y_train)
                left_ent = calc_entropy(left)
                right_ent = calc_entropy(right)
                
                gain = total - (len(left)/len(y_train) * left_ent + 
                               len(right)/len(y_train) * right_ent)
                gains.append(gain)
            
            best = np.argsort(gains)[-(n_bins-1):]
            thresholds = np.sort(thresholds[best])
        
        # make bins
        train_bins = np.digitize(X_train.flatten(), thresholds)
        test_bins = np.digitize(X_test.flatten(), thresholds)
        
        purity = calc_purity(train_bins, y_train)
        method = 'tree'
        
    except:
        # fallback: just use quantiles
        q = np.linspace(0, 1, n_bins + 1)[1:-1]
        thresholds = np.quantile(X_train, q)
        thresholds = np.unique(thresholds)
        
        train_bins = np.digitize(X_train.flatten(), thresholds)
        test_bins = np.digitize(X_test.flatten(), thresholds)
        
        purity = calc_purity(train_bins, y_train)
        method = 'quantile'
    
    return train_bins, test_bins, thresholds, method, purity

In [ ]:
# discretize everything
feature_cols = [c for c in train_features.columns if c != 'label']
y_train = train_features['label'].values

train_disc = pd.DataFrame()
test_disc = pd.DataFrame()
bin_info = {}

print("Discretizing features...")

for feat in feature_cols:
    X_train = train_features[feat].values
    X_test = test_features[feat].values
    
    train_bins, test_bins, thresh, method, purity = discretize(
        X_train, y_train, X_test
    )
    
    train_disc[feat] = train_bins
    test_disc[feat] = test_bins
    
    bin_info[feat] = {
        'thresholds': thresh.tolist() if thresh is not None else [],
        'n_bins': len(np.unique(train_bins)),
        'method': method,
        'purity': purity
    }
    
    print(f"{feat:25s} -> {len(np.unique(train_bins))} bins ({method})")

train_disc['label'] = y_train
test_disc['label'] = test_features['label'].values

print(f"\nDone: {train_disc.shape}")
# save bin info
with open('../output/bin_info.pkl', 'wb') as f:
    pickle.dump(bin_info, f)



Discretizing features...
burstiness                -> 5 bins (tree)
noun_ratio                -> 5 bins (tree)
verb_ratio                -> 5 bins (tree)
adj_ratio                 -> 5 bins (tree)
adv_ratio                 -> 5 bins (tree)
pos_diversity             -> 5 bins (tree)
syntactic_complexity      -> 5 bins (tree)
pronouns                  -> 5 bins (tree)
contractions              -> 5 bins (tree)
hapax                     -> 5 bins (tree)
formal_transitions        -> 5 bins (tree)
passive                   -> 5 bins (tree)
ngrams                    -> 5 bins (tree)
style_breaks              -> 4 bins (tree)
lexical_density           -> 5 bins (tree)

Done: (289263, 16)


In [9]:
# check what it looks like
print(train_disc.head())

print(f"\nValue range: {train_disc.drop('label', axis=1).min().min()} to "
      f"{train_disc.drop('label', axis=1).max().max()}")

   burstiness  noun_ratio  verb_ratio  adj_ratio  adv_ratio  pos_diversity  \
0           4           1           4          2          4              2   
1           4           0           4          0          4              1   
2           0           0           4          0          1              1   
3           0           4           4          0          4              4   
4           0           0           4          0          4              1   

   syntactic_complexity  pronouns  contractions  hapax  formal_transitions  \
0                     0         3             3      2                   0   
1                     0         4             4      1                   2   
2                     0         4             2      3                   0   
3                     0         1             4      3                   0   
4                     0         4             1      1                   0   

   passive  ngrams  style_breaks  lexical_density  label  
0  

In [10]:
# map bins to readable labels
def bin_to_label(bin_value, n_bins=5):
    """convert bin number to text label"""
    if n_bins == 5:
        labels = {0: 'very_low', 1: 'low', 2: 'medium', 3: 'high', 4: 'very_high'}
    elif n_bins == 3:
        labels = {0: 'low', 1: 'medium', 2: 'high'}
    else:
        return str(bin_value)
    return labels.get(bin_value, str(bin_value))

# convert all features to text labels
train_disc_text = train_disc.copy()
test_disc_text = test_disc.copy()

for feat in feature_cols:
    train_disc_text[feat] = train_disc[feat].apply(bin_to_label)
    test_disc_text[feat] = test_disc[feat].apply(bin_to_label)

print("Sample with text labels:")
print(train_disc_text.head())

Sample with text labels:
  burstiness noun_ratio verb_ratio adj_ratio  adv_ratio pos_diversity  \
0  very_high        low  very_high    medium  very_high        medium   
1  very_high   very_low  very_high  very_low  very_high           low   
2   very_low   very_low  very_high  very_low        low           low   
3   very_low  very_high  very_high  very_low  very_high     very_high   
4   very_low   very_low  very_high  very_low  very_high           low   

  syntactic_complexity   pronouns contractions   hapax formal_transitions  \
0             very_low       high         high  medium           very_low   
1             very_low  very_high    very_high     low             medium   
2             very_low  very_high       medium    high           very_low   
3             very_low        low    very_high    high           very_low   
4             very_low  very_high          low     low           very_low   

    passive     ngrams style_breaks lexical_density  label  
0       low 

In [11]:
# save both numeric and text versions
train_disc.to_csv('../output/discretized_train.csv', index=False)
test_disc.to_csv('../output/discretized_test.csv', index=False)

train_disc_text.to_csv('../output/discretized_train_text.csv', index=False)
test_disc_text.to_csv('../output/discretized_test_text.csv', index=False)



In [12]:
from collections import defaultdict
from itertools import combinations


In [13]:
# convert df to transactions
def get_transactions(df):
    transactions = []
    for idx, row in df.iterrows():
        transaction = set()
        for col in df.columns:
            item = f"{col}={row[col]}"
            transaction.add(item)
        transactions.append(transaction)
    return transactions

# calculate support
def get_support(itemset, transactions):
    count = sum(1 for t in transactions if itemset.issubset(t))
    return count / len(transactions) if transactions else 0

# find frequent 1-itemsets
def get_frequent_1(transactions, min_support):
    item_counts = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1
    
    n = len(transactions)
    return {
        frozenset([item]): count/n 
        for item, count in item_counts.items() 
        if count/n >= min_support
    }

In [14]:
def apriori_gen(prev_itemsets, k):
     #generate candidate k-itemsets
    candidates = set()
    itemsets = list(prev_itemsets.keys())
    
    for i in range(len(itemsets)):
        for j in range(i + 1, len(itemsets)):
            union = itemsets[i] | itemsets[j]
            if len(union) == k:
                candidates.add(union)
    
    # prune
    pruned = set()
    for candidate in candidates:
        subsets = [frozenset(s) for s in combinations(candidate, k-1)]
        if all(subset in prev_itemsets for subset in subsets):
            pruned.add(candidate)
    
    return pruned

def apriori(transactions, min_support, max_len=3):
     #apriori algorithm
    frequent = get_frequent_1(transactions, min_support)
    all_frequent = frequent.copy()
    
    k = 2
    while frequent and k <= max_len:
        candidates = apriori_gen(frequent, k)
        if not candidates:
            break
        
        frequent = {}
        for candidate in candidates:
            support = get_support(candidate, transactions)
            if support >= min_support:
                frequent[candidate] = support
        
        if frequent:
            all_frequent.update(frequent)
        
        k += 1
    
    return all_frequent

In [15]:
def generate_rules(frequent_itemsets, min_confidence, max_rules=50):
     #generate association rules
    # only use 2-itemsets and 3-itemsets
    filtered = {
        itemset: supp 
        for itemset, supp in frequent_itemsets.items() 
        if 2 <= len(itemset) <= 3
    }
    
    all_rules = []
    
    for itemset, itemset_support in filtered.items():
        items = list(itemset)
        
        for r in range(1, len(items)):
            for consequent_items in combinations(items, r):
                consequent = frozenset(consequent_items)
                antecedent = itemset - consequent
                
                if not antecedent:
                    continue
                
                ant_support = frequent_itemsets.get(antecedent, 0)
                if ant_support == 0:
                    continue
                
                confidence = itemset_support / ant_support
                
                if confidence < min_confidence:
                    continue
                
                cons_support = frequent_itemsets.get(consequent, 0)
                lift = confidence / cons_support if cons_support > 0 else 0
                
                if lift <= 1.0:
                    continue
                
                all_rules.append({
                    'antecedent': antecedent,
                    'consequent': consequent,
                    'support': itemset_support,
                    'confidence': confidence,
                    'lift': lift,
                })
    
    # remove redundant rules
    rules_by_cons = defaultdict(list)
    for rule in all_rules:
        rules_by_cons[rule['consequent']].append(rule)
    
    pruned = []
    for cons, cons_rules in rules_by_cons.items():
        cons_rules.sort(key=lambda r: (len(r['antecedent']), -r['confidence']))
        
        kept = []
        for rule in cons_rules:
            is_redundant = False
            for simpler in kept:
                if (len(simpler['antecedent']) < len(rule['antecedent']) and
                    simpler['antecedent'].issubset(rule['antecedent']) and
                    abs(simpler['confidence'] - rule['confidence']) < 0.05):
                    is_redundant = True
                    break
            if not is_redundant:
                kept.append(rule)
        
        pruned.extend(kept)
    
    # keep top N by quality
    for rule in pruned:
        rule['quality'] = rule['confidence'] * rule['lift'] * rule['support']
    
    pruned.sort(key=lambda r: r['quality'], reverse=True)
    
    return pruned[:max_rules]

In [16]:
# settings per class
thresholds = {
    0: {'support': 0.03, 'confidence': 0.50, 'max_rules': 50},
    1: {'support': 0.03, 'confidence': 0.50, 'max_rules': 50},
    2: {'support': 0.03, 'confidence': 0.50, 'max_rules': 50},
    3: {'support': 0.01, 'confidence': 0.40, 'max_rules': 50},
    4: {'support': 0.02, 'confidence': 0.35, 'max_rules': 50},
    5: {'support': 0.10, 'confidence': 0.30, 'max_rules': 50},
}

class_names = {
    0: "Fully Human",
    1: "Human-AI Polished",
    2: "AI-AI Humanized",
    3: "Human-AI Continued",
    4: "Deeply Mixed",
    5: "AI-Human Edited"
}

In [17]:
# mine rules per class
all_rules = []
y_train = train_disc['label'].values
train_features = train_disc.drop('label', axis=1)

print("Mining rules per class...\n")

for class_id in range(6):
    print(f"Class {class_id} ({class_names[class_id]}):")
    
    # get samples for this class
    mask = (y_train == class_id)
    df_class = train_features[mask].copy()
    n_samples = len(df_class)
    
    config = thresholds[class_id]
    min_support = config['support']
    min_confidence = config['confidence']
    max_rules = config['max_rules']
    
    print(f"  Samples: {n_samples:,}")
    print(f"  Support: {min_support:.0%}, Confidence: {min_confidence:.0%}")
    
    # convert to transactions
    transactions = get_transactions(df_class)
    
    # run apriori
    frequent = apriori(transactions, min_support, max_len=3)
    print(f"  Frequent itemsets: {len(frequent)}")
    
    if not frequent:
        print(f"  No rules found\n")
        continue
    
    # generate rules
    rules = generate_rules(frequent, min_confidence, max_rules)
    print(f"  Rules: {len(rules)}")
    
    # add metadata
    for rule in rules:
        rule['class'] = class_id
        rule['class_name'] = class_names[class_id]
        rule['support_count'] = int(rule['support'] * n_samples)
    
    all_rules.extend(rules)
    
    # show top 3
    print(f"  Top 3 rules:")
    for i, rule in enumerate(rules[:3], 1):
        ant = ', '.join(sorted(rule['antecedent']))
        cons = ', '.join(sorted(rule['consequent']))
        print(f"    {i}. {{{ant}}} -> {{{cons}}}")
        print(f"       Conf: {rule['confidence']:.0%}, Lift: {rule['lift']:.1f}x")
    print()

print(f"Total rules mined: {len(all_rules)}")

Mining rules per class...

Class 0 (Fully Human):
  Samples: 70,080
  Support: 3%, Confidence: 50%
  Frequent itemsets: 4324
  Rules: 50
  Top 3 rules:
    1. {formal_transitions=0} -> {syntactic_complexity=0}
       Conf: 94%, Lift: 1.0x
    2. {verb_ratio=4} -> {syntactic_complexity=0}
       Conf: 96%, Lift: 1.0x
    3. {syntactic_complexity=0} -> {formal_transitions=0}
       Conf: 79%, Lift: 1.0x

Class 1 (Human-AI Polished):
  Samples: 86,150
  Support: 3%, Confidence: 50%
  Frequent itemsets: 4561
  Rules: 50
  Top 3 rules:
    1. {verb_ratio=4} -> {syntactic_complexity=0}
       Conf: 97%, Lift: 1.0x
    2. {noun_ratio=0} -> {lexical_density=0}
       Conf: 90%, Lift: 2.3x
    3. {noun_ratio=0} -> {lexical_density=0, syntactic_complexity=0}
       Conf: 88%, Lift: 2.3x

Class 2 (AI-AI Humanized):
  Samples: 81,095
  Support: 3%, Confidence: 50%
  Frequent itemsets: 4534
  Rules: 50
  Top 3 rules:
    1. {verb_ratio=4} -> {syntactic_complexity=0}
       Conf: 95%, Lift: 1.0x
   

In [18]:
# save rules
with open('../output/association_rules.pkl', 'wb') as f:
    pickle.dump(all_rules, f)

print(f"Saved {len(all_rules)} rules to association_rules.pkl")

Saved 300 rules to association_rules.pkl


In [19]:
# load association rules
import pickle

with open('../output/association_rules.pkl', 'rb') as f:
    all_rules = pickle.load(f)

print(f"Loaded {len(all_rules)} association rules")

Loaded 300 association rules


In [25]:
# extract unique patterns from rules
# (antecedent, consequent, and their union)
all_patterns = set()

for rule in all_rules:
    all_patterns.add(rule['antecedent'])
    all_patterns.add(rule['consequent'])
    all_patterns.add(rule['antecedent'] | rule['consequent'])

print(f"Extracted {len(all_patterns)} unique patterns")

Extracted 172 unique patterns


In [23]:
# build transactions for all classes
transactions_by_class = {}

for class_id in range(6):
    mask = (y_train == class_id)
    df_class = train_disc[mask].drop('label', axis=1)
    
    transactions = []
    for idx, row in df_class.iterrows():
        transaction = set()
        for col in df_class.columns:
            transaction.add(f"{col}={row[col]}")
        transactions.append(transaction)
    
    transactions_by_class[class_id] = transactions
    print(f"Class {class_id}: {len(transactions):,} transactions")

print("Transactions built")

Class 0: 70,080 transactions
Class 1: 86,150 transactions
Class 2: 81,095 transactions
Class 3: 38,328 transactions
Class 4: 12,108 transactions
Class 5: 1,502 transactions
Transactions built


In [26]:
# calculate support for each pattern in each class
pattern_support = {}

print(f"Calculating support for {len(all_patterns)} patterns...")

for i, pattern in enumerate(all_patterns):
    if i % 100 == 0:
        print(f"  {i}/{len(all_patterns)}...", end='\r')
    
    pattern_support[pattern] = {}
    
    for class_id in range(6):
        transactions = transactions_by_class[class_id]
        count = sum(1 for t in transactions if pattern.issubset(t))
        support = count / len(transactions) if transactions else 0
        pattern_support[pattern][class_id] = support

print(f"  {len(all_patterns)}/{len(all_patterns)} done")

Calculating support for 172 patterns...
  172/172 done


In [28]:
# create classification rules
classification_rules = []

for pattern, class_supports in pattern_support.items():
    # find best class
    best_class = max(class_supports.items(), key=lambda x: x[1])[0]
    best_support = class_supports[best_class]
    
    # skip if support too low
    if best_support < 0.05:
        continue
    
    # calculate lift
    other_supports = [s for c, s in class_supports.items() if c != best_class]
    avg_other = np.mean(other_supports) if other_supports else 0.0001
    lift = best_support / avg_other if avg_other > 0 else 0
    
    # skip if not distinctive
    if lift < 1.5:
        continue
    
    # calculate confidence
    total = sum(class_supports.values())
    confidence = best_support / total if total > 0 else 0
    
    classification_rules.append({
        'pattern': pattern,
        'predicts_class': best_class,
        'class_name': class_names[best_class],
        'support_in_class': best_support,
        'support_in_others': avg_other,
        'lift': lift,
        'confidence': confidence,
        'pattern_size': len(pattern)
    })

print(f"Created {len(classification_rules)} classification rules")

Created 149 classification rules


In [29]:
# summary by class
print("\nClassification rules by class:")
for class_id in range(6):
    count = sum(1 for r in classification_rules if r['predicts_class'] == class_id)
    print(f"  Class {class_id} ({class_names[class_id]}): {count} rules")

print(f"\nTotal: {len(classification_rules)} classification rules")


Classification rules by class:
  Class 0 (Fully Human): 10 rules
  Class 1 (Human-AI Polished): 26 rules
  Class 2 (AI-AI Humanized): 29 rules
  Class 3 (Human-AI Continued): 6 rules
  Class 4 (Deeply Mixed): 57 rules
  Class 5 (AI-Human Edited): 21 rules

Total: 149 classification rules


In [30]:
# show example rules
print("\nExample classification rules:\n")

for class_id in [0, 2]:
    class_rules = [r for r in classification_rules if r['predicts_class'] == class_id]
    if not class_rules:
        continue
    
    class_rules.sort(key=lambda r: r['lift'], reverse=True)
    
    print(f"Class {class_id} ({class_names[class_id]}):")
    for rule in class_rules[:2]:
        pattern_str = ', '.join(sorted(rule['pattern']))
        print(f"  IF {{{pattern_str}}}")
        print(f"  THEN Class {class_id}")
        print(f"    {rule['support_in_class']*100:.1f}% in class, {rule['support_in_others']*100:.1f}% in others")
        print(f"    Lift: {rule['lift']:.1f}x, Confidence: {rule['confidence']*100:.0f}%\n")


Example classification rules:

Class 0 (Fully Human):
  IF {adj_ratio=0, pos_diversity=0, verb_ratio=0}
  THEN Class 0
    5.7% in class, 1.5% in others
    Lift: 3.9x, Confidence: 44%

  IF {adj_ratio=0, verb_ratio=0}
  THEN Class 0
    6.1% in class, 1.7% in others
    Lift: 3.6x, Confidence: 42%

Class 2 (AI-AI Humanized):
  IF {adj_ratio=4, lexical_density=4, noun_ratio=4}
  THEN Class 2
    19.4% in class, 6.1% in others
    Lift: 3.2x, Confidence: 39%

  IF {lexical_density=4, noun_ratio=4, style_breaks=0}
  THEN Class 2
    23.2% in class, 7.5% in others
    Lift: 3.1x, Confidence: 38%



In [31]:
# save classification rules
with open('../output/classification_rules.pkl', 'wb') as f:
    pickle.dump(classification_rules, f)

print(f"Saved {len(classification_rules)} rules to classification_rules.pkl")

Saved 149 rules to classification_rules.pkl


In [34]:
# load classification rules
import pickle
from collections import defaultdict, Counter

with open('../output/classification_rules.pkl', 'rb') as f:
    classification_rules = pickle.load(f)

print(f"Loaded {len(classification_rules)} classification rules")
print(f"\nRules per class:")
for cls in range(6):
    count = sum(1 for r in classification_rules if r['predicts_class'] == cls)
    print(f"  Class {cls}: {count} rules")

Loaded 149 classification rules

Rules per class:
  Class 0: 10 rules
  Class 1: 26 rules
  Class 2: 29 rules
  Class 3: 6 rules
  Class 4: 57 rules
  Class 5: 21 rules


In [35]:
# extract feature patterns from rules
class_patterns = {i: [] for i in range(6)}

for rule in classification_rules:
    cls = rule['predicts_class']
    pattern = rule['pattern']
    
    # extract features from pattern
    for item in pattern:
        if '=' in item:
            class_patterns[cls].append(item)

print("Extracted feature patterns from rules")

Extracted feature patterns from rules


In [36]:
# analyze most common patterns per class
class_names = {
    0: "Fully Human",
    1: "Human-AI Polished",
    2: "AI-AI Humanized",
    3: "Human-AI Continued",
    4: "Deeply Mixed",
    5: "AI-Human Edited"
}

print("\n" + "="*80)
print("FEATURE PROFILES FROM CLASSIFICATION RULES")
print("="*80)

for cls in range(6):
    print(f"\n{'='*80}")
    print(f"CLASS {cls}: {class_names[cls]}")
    print(f"{'='*80}")
    
    patterns = class_patterns[cls]
    
    if not patterns:
        print("  No rules for this class")
        continue
    
    # count pattern frequency
    pattern_counts = Counter(patterns)
    
    print(f"Total patterns: {len(patterns)}")
    print(f"\nTop 10 most common patterns:")
    
    for pattern, count in pattern_counts.most_common(10):
        pct = (count / len(patterns)) * 100
        print(f"  {pattern:35s} appears in {count:3d} rules ({pct:5.1f}%)")


FEATURE PROFILES FROM CLASSIFICATION RULES

CLASS 0: Fully Human
Total patterns: 21

Top 10 most common patterns:
  formal_transitions=0                appears in   4 rules ( 19.0%)
  burstiness=4                        appears in   3 rules ( 14.3%)
  noun_ratio=0                        appears in   3 rules ( 14.3%)
  hapax=0                             appears in   2 rules (  9.5%)
  syntactic_complexity=0              appears in   2 rules (  9.5%)
  adj_ratio=0                         appears in   2 rules (  9.5%)
  verb_ratio=0                        appears in   2 rules (  9.5%)
  ngrams=4                            appears in   1 rules (  4.8%)
  lexical_density=0                   appears in   1 rules (  4.8%)
  pos_diversity=0                     appears in   1 rules (  4.8%)

CLASS 1: Human-AI Polished
Total patterns: 63

Top 10 most common patterns:
  pos_diversity=0                     appears in  13 rules ( 20.6%)
  noun_ratio=0                        appears in   8 rules (

In [37]:
# analyze by feature (not feature=value)


for cls in range(6):
    print(f"\n{'-'*60}")
    print(f"Class {cls}: {class_names[cls]}")
    print(f"{'-'*60}")
    
    patterns = class_patterns[cls]
    
    if not patterns:
        print("  No rules")
        continue
    
    # extract just feature names
    features = []
    for pattern in patterns:
        if '=' in pattern:
            feat = pattern.split('=')[0]
            features.append(feat)
    
    feature_counts = Counter(features)
    
    print(f"Most important features (appear in most rules):")
    for feat, count in feature_counts.most_common(7):
        pct = (count / len(patterns)) * 100
        print(f"  {feat:25s}: {count:3d} times ({pct:5.1f}%)")


------------------------------------------------------------
Class 0: Fully Human
------------------------------------------------------------
Most important features (appear in most rules):
  formal_transitions       :   4 times ( 19.0%)
  burstiness               :   3 times ( 14.3%)
  noun_ratio               :   3 times ( 14.3%)
  hapax                    :   2 times (  9.5%)
  syntactic_complexity     :   2 times (  9.5%)
  adj_ratio                :   2 times (  9.5%)
  verb_ratio               :   2 times (  9.5%)

------------------------------------------------------------
Class 1: Human-AI Polished
------------------------------------------------------------
Most important features (appear in most rules):
  pos_diversity            :  16 times ( 25.4%)
  lexical_density          :  12 times ( 19.0%)
  noun_ratio               :  10 times ( 15.9%)
  adv_ratio                :   8 times ( 12.7%)
  pronouns                 :   8 times ( 12.7%)
  adj_ratio                :   5 t

In [5]:
# analyze high vs low values per feature per class

# map bin numbers to labels
bin_labels = {
    0: 'very_low',
    1: 'low', 
    2: 'medium',
    3: 'high',
    4: 'very_high'
}

for cls in range(6):
    print(f"\n{'='*70}")
    print(f"CLASS {cls}: {class_names[cls]}")
    print(f"{'='*70}")
    
    patterns = class_patterns[cls]
    
    if not patterns:
        print("  No rules")
        continue
    
    # group by feature
    feature_values = defaultdict(list)
    for pattern in patterns:
        if '=' in pattern:
            feat, val = pattern.split('=')
            try:
                bin_num = int(val)
                feature_values[feat].append(bin_num)
            except:
                pass
    
    # analyze each feature
    print("\nCharacteristic patterns:")
    for feat, values in sorted(feature_values.items()):
        avg_bin = sum(values) / len(values)
        most_common = Counter(values).most_common(1)[0]
        
        # determine if mostly high or low
        if avg_bin >= 3:
            tendency = "HIGH"
        elif avg_bin <= 1:
            tendency = "LOW"
        else:
            tendency = "MEDIUM"
        
        most_common_label = bin_labels.get(most_common[0], str(most_common[0]))
        
        print(f"  {feat:25s}: {tendency:6s} (avg bin={avg_bin:.1f}, "
              f"most common={most_common_label})")


CLASS 0: Fully Human

Characteristic patterns:
  adj_ratio                : LOW    (avg bin=0.0, most common=very_low)
  burstiness               : HIGH   (avg bin=4.0, most common=very_high)
  formal_transitions       : LOW    (avg bin=0.0, most common=very_low)
  hapax                    : LOW    (avg bin=0.0, most common=very_low)
  lexical_density          : LOW    (avg bin=0.0, most common=very_low)
  ngrams                   : HIGH   (avg bin=4.0, most common=very_high)
  noun_ratio               : LOW    (avg bin=0.0, most common=very_low)
  pos_diversity            : LOW    (avg bin=0.0, most common=very_low)
  syntactic_complexity     : LOW    (avg bin=0.0, most common=very_low)
  verb_ratio               : LOW    (avg bin=0.0, most common=very_low)

CLASS 1: Human-AI Polished

Characteristic patterns:
  adj_ratio                : LOW    (avg bin=0.0, most common=very_low)
  adv_ratio                : LOW    (avg bin=1.0, most common=very_low)
  formal_transitions       : LOW

In [7]:
# create readable class profiles

profiles = {}

for cls in range(6):
    patterns = class_patterns[cls]
    
    if not patterns:
        profiles[cls] = ["No distinctive patterns found"]
        continue
    
    # group by feature
    feature_values = defaultdict(list)
    for pattern in patterns:
        if '=' in pattern:
            feat, val = pattern.split('=')
            try:
                bin_num = int(val)
                feature_values[feat].append(bin_num)
            except:
                pass
    
    # create profile
    profile = []
    for feat, values in feature_values.items():
        avg_bin = sum(values) / len(values)
        
        if avg_bin >= 3.5:
            profile.append(f"Very HIGH {feat}")
        elif avg_bin >= 2.5:
            profile.append(f"HIGH {feat}")
        elif avg_bin <= 0.5:
            profile.append(f"Very LOW {feat}")
        elif avg_bin <= 1.5:
            profile.append(f"LOW {feat}")
    
    # sort by most distinctive (furthest from medium=2)
    profile_with_scores = []
    for p in profile:
        feat = p.split()[-1]
        avg = sum(feature_values[feat]) / len(feature_values[feat])
        distance = abs(avg - 2.0)  # distance from medium
        profile_with_scores.append((p, distance))
    
    profile_sorted = sorted(profile_with_scores, key=lambda x: x[1], reverse=True)
    profiles[cls] = [p[0] for p in profile_sorted]

# display
for cls in range(6):
    print(f"\n{class_names[cls]} (Class {cls}):")
    
    rules_count = sum(1 for r in classification_rules if r['predicts_class'] == cls)
    print(f"  Based on {rules_count} classification rules")
    print(f"  Characteristic features:")
    
    for i, characteristic in enumerate(profiles[cls][:5], 1):
        print(f"    {i}. {characteristic}")
    
    if not profiles[cls]:
        print("    (No distinctive patterns)")


Fully Human (Class 0):
  Based on 10 classification rules
  Characteristic features:
    1. Very LOW lexical_density
    2. Very LOW formal_transitions
    3. Very LOW noun_ratio
    4. Very LOW hapax
    5. Very HIGH burstiness

Human-AI Polished (Class 1):
  Based on 26 classification rules
  Characteristic features:
    1. Very LOW adj_ratio
    2. Very LOW syntactic_complexity
    3. Very LOW pronouns
    4. Very LOW formal_transitions
    5. Very LOW style_breaks

AI-AI Humanized (Class 2):
  Based on 29 classification rules
  Characteristic features:
    1. Very HIGH verb_ratio
    2. Very HIGH lexical_density
    3. Very LOW burstiness
    4. Very LOW style_breaks
    5. Very HIGH adj_ratio

Human-AI Continued (Class 3):
  Based on 6 classification rules
  Characteristic features:
    1. Very LOW contractions
    2. Very LOW syntactic_complexity
    3. Very LOW pronouns
    4. Very HIGH noun_ratio
    5. Very HIGH verb_ratio

Deeply Mixed (Class 4):
  Based on 57 classification

In [8]:
# human vs AI comparison based on rules


human_class = 0  # Fully Human
ai_class = 2     # AI-AI Humanized

print(f"\nComparing Class {human_class} ({class_names[human_class]}) vs "
      f"Class {ai_class} ({class_names[ai_class]}):\n")

# get feature values for each
human_features = defaultdict(list)
ai_features = defaultdict(list)

for pattern in class_patterns[human_class]:
    if '=' in pattern:
        feat, val = pattern.split('=')
        try:
            human_features[feat].append(int(val))
        except:
            pass

for pattern in class_patterns[ai_class]:
    if '=' in pattern:
        feat, val = pattern.split('=')
        try:
            ai_features[feat].append(int(val))
        except:
            pass

# compare
common_features = set(human_features.keys()) & set(ai_features.keys())

print(f"{'Feature':<25} {'Human Avg':<12} {'AI Avg':<12} {'Difference':<15}")
print("-" * 70)

for feat in sorted(common_features):
    human_avg = sum(human_features[feat]) / len(human_features[feat])
    ai_avg = sum(ai_features[feat]) / len(ai_features[feat])
    diff = ai_avg - human_avg
    
    if abs(diff) > 0.5:  # meaningful difference
        direction = "AI HIGHER" if diff > 0 else "AI LOWER"
        print(f"{feat:<25} {human_avg:<12.2f} {ai_avg:<12.2f} {diff:+.2f}  ({direction})")



Comparing Class 0 (Fully Human) vs Class 2 (AI-AI Humanized):

Feature                   Human Avg    AI Avg       Difference     
----------------------------------------------------------------------
adj_ratio                 0.00         4.00         +4.00  (AI HIGHER)
burstiness                4.00         0.00         -4.00  (AI LOWER)
lexical_density           0.00         4.00         +4.00  (AI HIGHER)
noun_ratio                0.00         4.00         +4.00  (AI HIGHER)
pos_diversity             0.00         4.00         +4.00  (AI HIGHER)
verb_ratio                0.00         4.00         +4.00  (AI HIGHER)


In [ ]:
# ACCURATE PROFILE
for cls in range(6):
    print(f"\n{class_names[cls]} (Class {cls}):")
    
    rules_count = sum(1 for r in classification_rules if r['predicts_class'] == cls)
    print(f"  Based on {rules_count} classification rules")
    print(f"  Common patterns include:")
    
    for i, characteristic in enumerate(profiles[cls][:5], 1):
        # Add nuance
        if cls in [2, 3] and ('verb_ratio' in characteristic or 'noun_ratio' in characteristic):
            print(f"    {i}. {characteristic} (in some rule patterns)")
        else:
            print(f"    {i}. {characteristic}")



Fully Human (Class 0):
  Based on 10 classification rules
  Common patterns include:
    1. Very LOW lexical_density
    2. Very LOW formal_transitions
    3. Very LOW noun_ratio
    4. Very LOW hapax
    5. Very HIGH burstiness

Human-AI Polished (Class 1):
  Based on 26 classification rules
  Common patterns include:
    1. Very LOW adj_ratio
    2. Very LOW syntactic_complexity
    3. Very LOW pronouns
    4. Very LOW formal_transitions
    5. Very LOW style_breaks

AI-AI Humanized (Class 2):
  Based on 29 classification rules
  Common patterns include:
    1. Very HIGH verb_ratio (in some rule patterns)
    2. Very HIGH lexical_density
    3. Very LOW burstiness
    4. Very LOW style_breaks
    5. Very HIGH adj_ratio

Human-AI Continued (Class 3):
  Based on 6 classification rules
  Common patterns include:
    1. Very LOW contractions
    2. Very LOW syntactic_complexity
    3. Very LOW pronouns
    4. Very HIGH noun_ratio (in some rule patterns)
    5. Very HIGH verb_ratio (in s